In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\adaly\OneDrive\Documents\DMDeficientGalaxyTNG-1")
CATALOG = PROJECT_DIR / "data" / "dmdg_catalog_z0.csv"

df = pd.read_csv(CATALOG)
print(f"Total Catalog Objects: {len(df):,}")
resolved = df[
    (df["N_star"] >= 500) &
    (df["SubhaloFlag"] == 1) &
    (df["f_DM"].notna())
].copy()

print(f"Resolved galaxies: {len(resolved):,}")
resolved["M_star_Msun"] = (
    resolved["M_star"] * 1e10 / 0.6774
)

mass_selected = resolved[
    (resolved["M_star_Msun"] > 1e9) &
    (resolved["M_star_Msun"] < 1e10)
].copy()
satellites = mass_selected[
    mass_selected["IsSatellite"] == True
].copy()
satellites["Host_M200c_Msun"] = (
    satellites["Host_M200c"] * 1e10 / 0.6774
)
group_selected = satellites[
    satellites["Host_M200c_Msun"] > 1e13
].copy()
dmdgs = group_selected[
    group_selected["f_DM"] < 0.5
].copy()
dmdgs_non_extreme = dmdgs[
   dmdgs["f_DM"] > 0.05
].copy()
print(f"DMDGS (non-tidal stripping): {len(dmdgs_non_extreme):,}")

Total Catalog Objects: 13,922,457
Resolved galaxies: 154,638
DMDGS (non-tidal stripping): 400


Imports and Generate Catalog

In [ ]:
features = ["M_total_2Rh", "f_DM", "M_star_Msun", "r_over_R200c"]
rank_cols = []
for col in features:
    median_val = dmdgs_non_extreme[col].median()
    dist_col = f"{col}_dist"
    dmdgs_non_extreme[dist_col] = (dmdgs_non_extreme[col] - median_val).abs()
    rank_col = f"{col}_rank"
    dmdgs_non_extreme[rank_col] = dmdgs_non_extreme[dist_col].rank(method="min")
    rank_cols.append(rank_col)
dmdgs_non_extreme["average_rank"] = dmdgs_non_extreme[rank_cols].mean(axis=1)
most_normal_sorted = dmdgs_non_extreme.sort_values(by="average_rank")
most_normal_galaxy = most_normal_sorted.iloc[0]
print("The most normal galaxy object has an average rank of:", most_normal_galaxy["average_rank"])
print(most_normal_galaxy)

The most normal galaxy object has an average rank of: 37.25
SubhaloID                           503372
GroupID                                473
M_DM_2Rh                          0.187221
M_total_2Rh                       0.532863
f_DM                               0.35135
M_star                            0.396336
M_gas                                  0.0
N_gas                                    0
N_DM                                    54
N_star                                 763
N_BH                                     0
R_half_star                       1.316552
x                                140710.44
y                                 76304.89
z                                57509.777
SFR                                    0.0
Vmax                            103.069984
VmaxRad                           1.513552
SubhaloFlag                           True
Host_M200c                       5492.0576
Host_R200c                       618.18994
IsCentral                            